# Phase 5: Model Training & Classification

This notebook focuses on training various multi-class classifiers to predict resume categories. 

### Data Leakage Prevention Protocol
1. **Load Raw Cleaned Data**: We bypass pre-generated feature matrices to ensure total control over the split.
2. **Pre-Split Stratification**: Split data into train (80%) and test (20%) immediately.
3. **Context-Aware Vectorization**: Fit TF-IDF on the training set only, then transform the test set.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline
sns.set_style("whitegrid")

## 1. Load Data

In [ ]:
# Load cleaned resumes
df = pd.read_csv("../data/processed/resumes_cleaned.csv")
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Rigorous Train/Test Split
We use stratification to ensure category distribution is preserved in both sets.

In [ ]:
X = df['cleaned_text'].fillna('')
y = df['Category']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training set: {X_train_raw.shape}")
print(f"Testing set:  {X_test_raw.shape}")

## 3. Label Encoding
Converting category strings to integers.

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

# Save label encoder for future use
os.makedirs("../models", exist_ok=True)
joblib.dump(le, "../models/label_encoder.pkl")

print("Classes:", le.classes_)

## 4. Feature Extraction (TF-IDF)
Fitting strictly on `X_train_raw` to prevent data leakage.

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

# Save vectorizer
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print(f"TF-IDF Train shape: {X_train.shape}")
print(f"TF-IDF Test shape:  {X_test.shape}")

## 5. Baseline Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42),
    "SGD Classifier": SGDClassifier(loss='hinge', penalty='l2', alpha=1e-3, random_state=42, max_iter=100),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("\n")

## 6. Compare Baseline Accuracies

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=list(results.keys()), y=list(results.values()))
plt.title('Baseline Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1.0)
plt.show()